# 训练循环与损失函数

> 上一节看了 MoE——大模型不只是一堆 Transformer Block，还可能有路由器、专家、辅助 loss 这些训练难点。这一节换一个角度，看一条样本是如何进入 Hugging Face Transformers、ModelScope ms-swift 这类工业训练库，最后变成 loss 和参数更新的。
>
> 我们会先用极小数字手算 loss，再写一个迷你 Trainer，最后把每一步映射到工业训练库的真实代码。

工业训练库里写一句 `trainer.train()`，模型就开始训练。这句代码背后发生的事情大致是这样：

```text
原始文本 / 对话数据
  ↓
Tokenizer 和 Chat Template
  ↓
input_ids / attention_mask / labels
  ↓
Data Collator 拼 batch
  ↓
model(**batch) 得到 logits 和 loss
  ↓
loss.backward()
  ↓
optimizer.step()
  ↓
lr_scheduler.step()
  ↓
保存 checkpoint / 打日志 / 评估
```

这一章会把这条链路从头走到尾。

先对文中提到的库名做说明：

| 生态 | 常见训练库 | 常见入口 |
|:---|:---|:---|
| 国际社区 | Hugging Face Transformers | `Trainer`、`TrainingArguments`、`DataCollatorForLanguageModeling` |
| 中国区 / ModelScope | ModelScope SWIFT，也叫 `ms-swift` | `swift sft`、`SftArguments`、LoRA/QLoRA/DeepSpeed 等封装 |

`ms-swift` 是 ModelScope 生态里的大模型训练工具箱，负责模型加载、数据集、Chat Template、SFT、LoRA、分布式训练、评测和部署。它不改变 Transformer 原理，而是把训练 loop 的工程细节封装起来。

In [1]:
# 本章所有 import 集中放在第一个 code cell
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
print("本章会用 PyTorch 模拟工业训练库背后的核心逻辑。")
print("关键观察：Trainer 不是魔法，它只是把数据处理、forward、loss、backward、step 串起来。")


本章会用 PyTorch 模拟工业训练库背后的核心逻辑。
关键观察：Trainer 不是魔法，它只是把数据处理、forward、loss、backward、step 串起来。


## 0. 先建立直觉：Trainer 到底帮你省了什么？

先定义三个词。

Trainer 是训练管理器，本身不是模型，而是反复执行训练步骤的程序。Hugging Face 的 `Trainer` 和 `ms-swift` 的 SFT 入口都属于这一类。

Batch 是一次送进模型的一小批样本。比如一张显卡一次处理 2 条对话，每条长度补齐到 8 个 token，batch 里的 `input_ids` 形状就是 `[2, 8]`。

Loss 是模型猜错的程度。语言模型最常用的是 Cross-Entropy Loss：正确 token 的概率越低，loss 越大。

把训练 loop 拆开看，是一组按顺序执行的步骤：

```text
1. 从 dataset 取出一批原始样本
2. tokenizer 把文本切成 token id
3. data collator 把多条样本 pad 成一个 batch
4. 模型 forward，得到 logits
5. 用 labels 算 cross-entropy loss
6. loss.backward() 算梯度
7. optimizer.step() 更新参数
8. 记录日志、跑评估、保存 checkpoint
```

PyTorch 里手写训练 loop，前 7 步核心代码大约 10 行。工业训练库之所以写几十上百行，是因为真实训练除了这 7 步，还要处理显存、混合精度、梯度累积、分布式、断点续训、评估、保存权重、LoRA 这些工程细节。

数学上的训练 loop 始终是同一句话：让模型看 input_ids，预测 labels，loss 反向传播，更新参数。

## 1. 一条文本样本如何变成训练样本？

我们先不用真实 Tokenizer，手动造一个极小词表。这样每个数字的含义一目了然。

```text
词表：
0=<pad>, 1=<bos>, 2=<eos>, 3=我, 4=爱, 5=机器, 6=学习

原句：我 爱 机器 学习
完整 token 序列：[<bos>, 我, 爱, 机器, 学习, <eos>]
数字形式：       [1,     3,  4, 5,   6,    2]
```

语言模型训练时，不是额外人工标注“答案”。答案就在原句的下一个 token 里：

```text
input_ids = [1, 3, 4, 5, 6]
labels    = [3, 4, 5, 6, 2]
```

也就是说：

```text
看到 <bos>             → 应该预测 我
看到 <bos> 我          → 应该预测 爱
看到 <bos> 我 爱       → 应该预测 机器
看到 <bos> 我 爱 机器  → 应该预测 学习
看到 <bos> 我 爱 机器 学习 → 应该预测 <eos>
```

这叫 **next-token prediction**：给前面的 token，预测下一个 token。


In [2]:
# 用具体数字构造 input_ids 和 labels
id_to_token = {
    0: "<pad>",
    1: "<bos>",
    2: "<eos>",
    3: "我",
    4: "爱",
    5: "机器",
    6: "学习",
}

sentence = torch.tensor([1, 3, 4, 5, 6, 2])
input_ids = sentence[:-1]
labels = sentence[1:]

print("完整句子:", sentence.tolist())
print("input_ids:", input_ids.tolist())
print("labels:   ", labels.tolist())
print()

for pos in range(len(input_ids)):
    context = [id_to_token[i.item()] for i in input_ids[:pos + 1]]
    target = id_to_token[labels[pos].item()]
    print(f"位置 {pos}: 看到 {context} -> 预测 {target}")

print()
print("关键观察：labels 不是人工额外写的答案，而是同一句话整体左移/右移得到的。")


完整句子: [1, 3, 4, 5, 6, 2]
input_ids: [1, 3, 4, 5, 6]
labels:    [3, 4, 5, 6, 2]

位置 0: 看到 ['<bos>'] -> 预测 我
位置 1: 看到 ['<bos>', '我'] -> 预测 爱
位置 2: 看到 ['<bos>', '我', '爱'] -> 预测 机器
位置 3: 看到 ['<bos>', '我', '爱', '机器'] -> 预测 学习
位置 4: 看到 ['<bos>', '我', '爱', '机器', '学习'] -> 预测 <eos>

关键观察：labels 不是人工额外写的答案，而是同一句话整体左移/右移得到的。


## 2. Chat Template 为什么会影响 loss？

普通预训练数据是一段连续文本，所有 token 都可以算 loss。但 SFT（监督微调）经常是对话数据：

```text
user: 2 + 2 等于几？
assistant: 4
```

这时有一个关键问题：模型应该学什么？

通常我们希望模型学习 **assistant 如何回答**，不希望它因为 user 的提问部分被惩罚。于是工业训练库会把 label 做成这样：

```text
input_ids: [<user>, 2, +, 2, ?, <assistant>, 4, <eos>]
labels:    [-100,  -100, -100, -100, -100, -100, 4, <eos>]
```

这里的 `-100` 是 PyTorch `cross_entropy` 的默认忽略标记。它的意思是：这个位置不参与 loss。

**Chat Template**：把结构化对话转成模型能读的文本格式的模板。比如把 `user` 和 `assistant` 加上特殊标记。不同模型的模板可能不同，Qwen、LLaMA、ChatGLM 都可能有自己的格式。

所以在 `ms-swift` 或 Transformers 里，SFT 的数据处理不只是“分词”，还包括：

```text
messages -> apply_chat_template -> tokenize -> labels mask
```


In [3]:
# 手动模拟一次 Chat Template 和 label mask
vocab = {
    "<pad>": 0,
    "<user>": 1,
    "<assistant>": 2,
    "<eos>": 3,
    "2": 4,
    "+": 5,
    "=": 6,
    "?": 7,
    "4": 8,
}

chat_tokens = ["<user>", "2", "+", "2", "?", "<assistant>", "4", "<eos>"]
input_ids = torch.tensor([vocab[token] for token in chat_tokens])

# 只训练 assistant 回复：位置 6 和 7 参与 loss，其余位置忽略
labels = torch.tensor([-100, -100, -100, -100, -100, -100, vocab["4"], vocab["<eos>"]])

print("tokens:   ", chat_tokens)
print("input_ids:", input_ids.tolist())
print("labels:   ", labels.tolist())
print()

for token, label in zip(chat_tokens, labels.tolist()):
    if label == -100:
        print(f"{token:>11s} -> 不算 loss")
    else:
        print(f"{token:>11s} -> 算 loss，目标 token id = {label}")

print()
print("关键观察：SFT 不是让模型复读 user，而是主要惩罚 assistant 回复部分。")


tokens:    ['<user>', '2', '+', '2', '?', '<assistant>', '4', '<eos>']
input_ids: [1, 4, 5, 4, 7, 2, 8, 3]
labels:    [-100, -100, -100, -100, -100, -100, 8, 3]

     <user> -> 不算 loss
          2 -> 不算 loss
          + -> 不算 loss
          2 -> 不算 loss
          ? -> 不算 loss
<assistant> -> 不算 loss
          4 -> 算 loss，目标 token id = 8
      <eos> -> 算 loss，目标 token id = 3

关键观察：SFT 不是让模型复读 user，而是主要惩罚 assistant 回复部分。


## 3. 手算 Cross-Entropy：loss 到底怎么算？

模型在每个位置输出的不是一个 token，而是一整排分数，叫 **logits**。

**Logits**：模型对词表里每个 token 打的原始分数，还不是概率。比如词表有 4 个 token，某个位置的 logits 可能是：

```text
[2.0, 1.0, 0.1, -1.0]
```

要算 loss，要先做 softmax：

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

然后只看正确 token 的概率 $p_{correct}$：

$$loss = -\log(p_{correct})$$

为什么取负 log？直觉上：

```text
正确概率 = 1.0  -> -log(1.0)  = 0      很好
正确概率 = 0.5  -> -log(0.5)  = 0.693  还行
正确概率 = 0.01 -> -log(0.01) = 4.605  很差
```


In [4]:
# 手算一个位置的 Cross-Entropy Loss
logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
correct_id = 0

exp_values = torch.exp(logits)
probs = exp_values / exp_values.sum()
correct_prob = probs[correct_id].item()
manual_loss = -math.log(correct_prob)

torch_loss = F.cross_entropy(logits.view(1, -1), torch.tensor([correct_id]))

print("logits:", logits.tolist())
print("exp(logits):", [round(x, 4) for x in exp_values.tolist()])
print("softmax 概率:", [round(x, 4) for x in probs.tolist()])
print()
print(f"正确 token id = {correct_id}")
print(f"正确 token 概率 = {correct_prob:.4f}")
print(f"手算 loss = -log({correct_prob:.4f}) = {manual_loss:.4f}")
print(f"PyTorch loss = {torch_loss.item():.4f}")
print()
print("关键观察：cross_entropy = log_softmax + 取正确类 + 负号。")


logits: [2.0, 1.0, 0.10000000149011612, -1.0]
exp(logits): [7.3891, 2.7183, 1.1052, 0.3679]
softmax 概率: [0.6381, 0.2347, 0.0954, 0.0318]

正确 token id = 0
正确 token 概率 = 0.6381
手算 loss = -log(0.6381) = 0.4493
PyTorch loss = 0.4493

关键观察：cross_entropy = log_softmax + 取正确类 + 负号。


## 4. 一个 batch 进入模型后发生了什么？

工业训练库不会一条一条训练，而是把多条样本拼成 batch。

问题来了：两条句子长度不同怎么办？

```text
样本 A: [<bos>, 我, 爱, 机器, 学习, <eos>]     长度 6
样本 B: [<bos>, 我, 爱, 学习, <eos>]           长度 5
```

要放进同一个张量，必须 pad 到一样长：

```text
input_ids:
A: [1, 3, 4, 5, 6]
B: [1, 3, 4, 6, 0]

labels:
A: [3, 4, 5, 6, 2]
B: [3, 4, 6, 2, -100]
```

`attention_mask` 告诉模型哪些位置是真 token，哪些是 pad：

```text
A: [1, 1, 1, 1, 1]
B: [1, 1, 1, 1, 0]
```

**Data Collator**：把多条已经 tokenize 的样本整理成一个 batch 的组件。它通常负责 padding、生成 `attention_mask`、整理 `labels`。


In [5]:
# 写一个极简 Data Collator，模拟工业训练库拼 batch 的过程
def simple_collate(features, pad_id=0, ignore_index=-100):
    """
    把多条样本补齐成一个 batch。

    参数:
        features: 每条样本包含 input_ids 和 labels
        pad_id: input_ids 使用的 padding token id
        ignore_index: labels 使用的忽略标记

    返回:
        一个 batch 字典，包含 input_ids、attention_mask、labels
    """
    max_len = max(len(item["input_ids"]) for item in features)

    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    for item in features:
        input_ids = item["input_ids"]
        labels = item["labels"]
        pad_len = max_len - len(input_ids)

        batch_input_ids.append(input_ids + [pad_id] * pad_len)
        batch_attention_mask.append([1] * len(input_ids) + [0] * pad_len)
        batch_labels.append(labels + [ignore_index] * pad_len)

    return {
        "input_ids": torch.tensor(batch_input_ids),
        "attention_mask": torch.tensor(batch_attention_mask),
        "labels": torch.tensor(batch_labels),
    }

features = [
    {"input_ids": [1, 3, 4, 5, 6], "labels": [3, 4, 5, 6, 2]},
    {"input_ids": [1, 3, 4, 6], "labels": [3, 4, 6, 2]},
]

batch = simple_collate(features)

for name, value in batch.items():
    print(f"{name}: shape={tuple(value.shape)}")
    print(value)
    print()

print("关键观察：input_ids 用 pad_id 补齐，labels 用 -100 补齐。")
print("如果 labels 也用 pad_id，模型就会被迫学习预测 <pad>，这是错误的。")


input_ids: shape=(2, 5)
tensor([[1, 3, 4, 5, 6],
        [1, 3, 4, 6, 0]])

attention_mask: shape=(2, 5)
tensor([[1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0]])

labels: shape=(2, 5)
tensor([[   3,    4,    5,    6,    2],
        [   3,    4,    6,    2, -100]])

关键观察：input_ids 用 pad_id 补齐，labels 用 -100 补齐。
如果 labels 也用 pad_id，模型就会被迫学习预测 <pad>，这是错误的。


## 5. 模型 forward：为什么 labels 可以直接传给 model？

在 Transformers 里常见写法是：

```python
outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
loss = outputs.loss
logits = outputs.logits
```

这不是说模型“需要答案才能预测”。真实顺序是：

```text
1. 模型用 input_ids 做 forward，得到 logits
2. 如果你传了 labels，模型内部顺手帮你算 cross_entropy loss
3. 返回 outputs.loss 和 outputs.logits
```

也就是说，`labels` 只用于算 loss，不会泄露给模型做预测。

下面我们写一个极小的 Causal LM。它没有真正的 Transformer Block，只用 Embedding + Linear 演示接口。重点不是模型强不强，而是训练库调用模型的形状和 loss 计算方式。


In [6]:
class TinyCausalLM(nn.Module):
    """
    一个极小的 Causal Language Model，用来演示 Trainer 调用模型的接口。

    参数:
        vocab_size: 词表大小
        hidden_size: token 向量维度
    """
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        输入 token ids，输出 logits；如果提供 labels，就同时计算 loss。

        参数:
            input_ids: [batch, seq_len]
            attention_mask: [batch, seq_len]，这里为了简化不参与计算
            labels: [batch, seq_len]，-100 的位置不算 loss

        返回:
            一个字典，包含 loss 和 logits
        """
        hidden = self.embedding(input_ids)
        logits = self.lm_head(hidden)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            )

        return {"loss": loss, "logits": logits}

model = TinyCausalLM(vocab_size=9, hidden_size=16)
outputs = model(**batch)

print("logits shape:", tuple(outputs["logits"].shape))
print("loss:", round(outputs["loss"].item(), 4))
print()
print("关键观察：logits 是 [batch, seq_len, vocab_size]。")
print("loss 是把所有非 -100 的位置摊平后一起算平均。")


logits shape: (2, 5, 9)
loss: 1.8008

关键观察：logits 是 [batch, seq_len, vocab_size]。
loss 是把所有非 -100 的位置摊平后一起算平均。


## 6. 实现一个迷你 Trainer：工业库的核心骨架

现在我们把 pieces 拼起来。一个最小训练 step 是：

```text
batch = next(dataloader)
outputs = model(**batch)
loss = outputs.loss
loss.backward()
optimizer.step()
optimizer.zero_grad()
```

但工业训练库还会多做几件事：

| 工程功能 | 为什么需要 |
|:---|:---|
| gradient accumulation | 显存不够大 batch，就多次小 batch 累积梯度 |
| lr scheduler | 学习率不能永远固定，通常先 warmup 再下降 |
| mixed precision | 用 FP16/BF16 加速并省显存 |
| gradient clipping | 防止梯度突然爆炸 |
| checkpoint | 训练中断后能恢复 |
| eval / logging | 知道模型有没有真的变好 |
| distributed training | 多 GPU / 多机器并行训练 |

我们先写一个不含分布式和混合精度的迷你版本。它和 `Trainer` 的核心没有本质区别。


In [7]:
class ToyTextDataset(Dataset):
    """
    一个极小文本数据集，每条样本已经是 token ids。

    参数:
        sequences: 多条完整 token 序列，每条包含 BOS/EOS
    """
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        ids = self.sequences[index]
        return {
            "input_ids": ids[:-1],
            "labels": ids[1:],
        }

sequences = [
    [1, 3, 4, 5, 6, 2],
    [1, 3, 4, 6, 2],
    [1, 3, 4, 5, 2],
    [1, 3, 4, 6, 2],
]

dataset = ToyTextDataset(sequences)
dataloader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=simple_collate)

model = TinyCausalLM(vocab_size=9, hidden_size=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

print("训练前，先看一个 batch：")
first_batch = next(iter(dataloader))
for name, value in first_batch.items():
    print(name, tuple(value.shape))
print()
print("关键观察：Dataset 负责单条样本，DataLoader + collator 负责拼 batch。")


训练前，先看一个 batch：
input_ids (2, 5)
attention_mask (2, 5)
labels (2, 5)

关键观察：Dataset 负责单条样本，DataLoader + collator 负责拼 batch。


In [8]:
# 迷你训练 loop：这就是 Trainer 最核心的部分
num_epochs = 8
log_every = 1

for epoch in range(num_epochs):
    total_loss = 0.0
    num_steps = 0

    for batch in dataloader:
        outputs = model(**batch)
        loss = outputs["loss"]

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        num_steps += 1

    avg_loss = total_loss / num_steps
    if (epoch + 1) % log_every == 0:
        print(f"epoch {epoch + 1:02d} | train_loss = {avg_loss:.4f}")

print()
print("关键观察：loss 下降不是因为 Trainer 有魔法，而是参数被 optimizer 一步步改了。")


epoch 01 | train_loss = 2.0311


epoch 02 | train_loss = 0.6100
epoch 03 | train_loss = 0.3348
epoch 04 | train_loss = 0.2968


epoch 05 | train_loss = 0.2741


epoch 06 | train_loss = 0.2674
epoch 07 | train_loss = 0.2618
epoch 08 | train_loss = 0.2511

关键观察：loss 下降不是因为 Trainer 有魔法，而是参数被 optimizer 一步步改了。


## 7. 优化器：拿到梯度之后，参数怎么更新？

> 上一节写了一个迷你训练 loop，里面有一行 `optimizer.step()`——模型参数在这一步被更新。但这个 step 里到底发生了什么、为什么是 AdamW 而不是朴素的 SGD，一直没有展开。
>
> 这一节从 Adam 的两条核心公式出发，把 optimizer.step() 内部的机制拆开：一阶矩、二阶矩、bias correction、weight decay、gradient clipping，最后走到 2024 年兴起的 Muon。重点不是堆数学推导，而是建立「每个优化器在解决什么问题」的直觉。

优化器决定一件事：拿到梯度之后，参数朝哪个方向走、走多远。朴素的 SGD 直接用 $\theta \leftarrow \theta - \eta g$ 更新，但在大模型这种几十亿参数、几万步训练的场景下，SGD 几乎无法收敛——梯度噪声大、不同参数方向的有效曲率差异大、初期更新方向不稳定。

Adam 引入一阶矩和二阶矩的指数滑动平均，相当于给每个参数单独算一个自适应学习率，成了过去十年 LLM 训练的事实标准。Adam 不是终点——2024 年 Keller Jordan 在 nanoGPT Speedrun 里用 Muon 把训练时间砍掉一半，二阶优化器（Shampoo、SOAP）重新回到视野。理解这条演进线，需要先把 Adam 本身的工程细节理清楚。


### 7.1 Adam：一阶矩 + 二阶矩

Adam 为每个参数维护两个状态：一阶矩 $m$（梯度的指数滑动平均，也叫 momentum）和二阶矩 $v$（梯度平方的指数滑动平均）。在第 $t$ 步，拿到梯度 $g_t$ 后：

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$

$m$ 的作用是方向累积：如果连续几步的梯度都指向同一方向，$m$ 会越来越大，相当于给这一方向加权；如果梯度来回震荡，正负抵消，$m$ 接近 0。这样在 noisy gradient 下也能稳定前进。

$v$ 的作用是逐元素归一化：每个参数位置有自己的 $v$，更新时分母是 $\sqrt{v} + \epsilon$。那些长期梯度很大的位置，$v$ 很大，更新幅度会被自动压小——这是自适应学习率的来源。

两个状态都从 0 开始初始化，所以前几步的估计偏小。Adam 用 bias correction 修正：

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

当 $t$ 很大时 $\beta^t \to 0$，修正项趋于 1，只在初期显著放大估计值。最终更新公式：

$$\theta_t = \theta_{t-1} - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

典型超参：$\beta_1 = 0.9$、$\beta_2 = 0.999$、$\epsilon = 10^{-8}$。LLM 预训练里学习率通常在 $10^{-4}$ 量级。$\beta_2$ 设到 0.999 是因为梯度二阶矩变化比一阶矩慢，需要更长的滑动窗口才能稳定估计。


#### 手算一个 Adam step

假设某个参数的初始值 $\theta_0 = 1.0$，第一步梯度 $g_1 = 0.5$，超参 $\beta_1 = 0.9$、$\beta_2 = 0.999$、$\eta = 0.1$、$\epsilon = 10^{-8}$。$m_0 = v_0 = 0$。

- $m_1 = 0.9 \cdot 0 + 0.1 \cdot 0.5 = 0.05$
- $v_1 = 0.999 \cdot 0 + 0.001 \cdot 0.25 = 0.00025$
- $\hat{m}_1 = 0.05 / (1 - 0.9) = 0.5$（等于 $g_1$，bias correction 把单步估计完全恢复）
- $\hat{v}_1 = 0.00025 / (1 - 0.999) = 0.25$（等于 $g_1^2$）
- $\Delta\theta = -0.1 \cdot 0.5 / (\sqrt{0.25} + 10^{-8}) = -0.1$
- $\theta_1 = 1.0 - 0.1 = 0.9$

关键观察：bias correction 让第一步的有效更新等于 $\eta \cdot \text{sign}(g_1)$，否则 $m_1 = 0.05$ 太小、$v_1 = 0.00025$ 也让分母很小，更新幅度会被严重低估。


In [9]:
import torch

torch.manual_seed(42)

# 手算验证：单个参数走一步 Adam
theta = torch.tensor([1.0], requires_grad=True)
theta.grad = torch.tensor([0.5])

optimizer = torch.optim.Adam([theta], lr=0.1, betas=(0.9, 0.999), eps=1e-8)
optimizer.step()

print(f"theta 更新前: 1.0")
print(f"theta 更新后: {theta.item():.6f}")
print(f"手算预期:     0.9")
print(f"误差:         {abs(theta.item() - 0.9):.2e}")
print()
print("关键观察：第一步 bias correction 让 Adam 更新幅度等于 lr * sign(grad)。")
print("如果不做 correction，更新幅度会被严重低估。")


theta 更新前: 1.0
theta 更新后: 0.900000
手算预期:     0.9
误差:         3.58e-08

关键观察：第一步 bias correction 让 Adam 更新幅度等于 lr * sign(grad)。
如果不做 correction，更新幅度会被严重低估。


#### 手写 SimpleAdam

理解 Adam 最快的方式是写一个能跑的最小实现。继承 `torch.optim.Optimizer`，在 `step()` 里按公式更新。这段代码和 PyTorch 官方实现的核心逻辑一致，只是省去了 AMP、foreach 等工程优化。


In [10]:
import torch


class SimpleAdam(torch.optim.Optimizer):
    """简化版 Adam，只保留核心逻辑，方便对照公式阅读"""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        defaults = {"lr": lr, "betas": betas, "eps": eps}
        super().__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["betas"]
            eps = group["eps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad.data
                state = self.state[p]

                # 初次访问时初始化状态
                if "t" not in state:
                    state["t"] = 0
                    state["m"] = torch.zeros_like(p.data)
                    state["v"] = torch.zeros_like(p.data)

                state["t"] += 1
                t = state["t"]
                m, v = state["m"], state["v"]

                # 一阶矩和二阶矩的指数滑动平均
                m.mul_(beta1).add_(grad, alpha=1 - beta1)
                v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # bias correction
                m_hat = m / (1 - beta1 ** t)
                v_hat = v / (1 - beta2 ** t)

                # 参数更新
                p.data.sub_(lr * m_hat / (v_hat.sqrt() + eps))


# 验证：和 PyTorch 官方 Adam 在同样输入下输出一致
torch.manual_seed(0)
p1 = torch.randn(100, requires_grad=True)
p2 = p1.clone().detach().requires_grad_(True)
grad = torch.randn(100) * 0.1
p1.grad = grad.clone()
p2.grad = grad.clone()

opt_torch = torch.optim.Adam([p1], lr=1e-3)
opt_ours = SimpleAdam([p2], lr=1e-3)
opt_torch.step()
opt_ours.step()

max_diff = (p1 - p2).abs().max().item()
print(f"PyTorch Adam vs SimpleAdam 第一步最大差异: {max_diff:.2e}")

# 再走 99 步
for _ in range(99):
    g = torch.randn(100) * 0.1
    p1.grad = g.clone()
    p2.grad = g.clone()
    opt_torch.step()
    opt_ours.step()

max_diff = (p1 - p2).abs().max().item()
print(f"100 步后最大差异: {max_diff:.2e}")
print("关键观察：手写实现和官方 Adam 数值一致，公式理解正确。")


PyTorch Adam vs SimpleAdam 第一步最大差异: 1.16e-10
100 步后最大差异: 2.98e-08
关键观察：手写实现和官方 Adam 数值一致，公式理解正确。


### 7.2 AdamW：解耦 weight decay

Adam 原始论文里如果要加 weight decay（L2 正则），做法是把 $\lambda \theta$ 加到梯度上：$g_t^{\text{reg}} = g_t + \lambda \theta$，然后让这个修改后的梯度进入 $m$、$v$ 的更新。

问题在于，L2 penalty 的梯度 $\lambda \theta$ 会经过 $m / \sqrt{v}$ 的归一化——weight decay 的强度被自适应学习率缩放了。那些历史梯度很大的参数，$\sqrt{v}$ 很大，weight decay 的实际效果就被削弱；历史梯度小的参数，weight decay 反而被放大。这破坏了 weight decay「均匀地把参数往 0 拉」的初衷。

AdamW（Loshchilov & Hutter, 2019）的做法是把 weight decay 从梯度链路里解耦出来：直接在参数上做衰减，不经过 $m / \sqrt{v}$。更新公式变成：

$$\theta_t = \theta_{t-1} - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} - \eta \lambda \theta_{t-1}$$

工程上通常把 decay 项放在 Adam update 之前（`p.data -= lr * weight_decay * p.data`），这样后面的 $\hat{m}/\sqrt{\hat{v}}$ 是基于衰减后的参数算的。


#### 哪些参数该 decay、哪些不该 decay

标准 recipe：bias 和 LayerNorm 参数不加 weight decay，其他（Linear weight、Embedding）加。原因是 bias 和 LayerNorm 的作用是平移激活值的均值，把它们往 0 拉没有意义甚至会损害表达能力。Linear weight 和 Embedding 是缩放型参数，weight decay 防止数值过大导致激活饱和。

PyTorch 里用 parameter group 区分：`p.dim() >= 2` 是一个常用的启发式——2D 及以上的 tensor 通常是 Linear/Conv/Embedding 的权重，1D 的通常是 bias 和 LayerNorm 的 $\gamma, \beta$。


In [11]:
import torch
import torch.nn as nn


# 一个极简的 decoder-only 模型骨架，用来演示参数分组
class TinyBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.ln = nn.LayerNorm(d)
        self.fc = nn.Linear(d, d * 4)
        self.proj = nn.Linear(d * 4, d)

    def forward(self, x):
        return self.proj(torch.nn.functional.gelu(self.fc(self.ln(x))))


model = nn.Sequential(
    nn.Embedding(1000, 64),
    TinyBlock(64),
    TinyBlock(64),
    nn.Linear(64, 1000),
)

# 按维度分组：2D 及以上 decay，1D 不 decay
decay, no_decay = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if p.dim() >= 2:
        decay.append(p)
    else:
        no_decay.append(p)

print(f"decay 参数数量:    {len(decay)}, 总元素: {sum(p.numel() for p in decay)}")
print(f"no_decay 参数数量: {len(no_decay)}, 总元素: {sum(p.numel() for p in no_decay)}")
print()
print("典型分布：decay 占绝大多数（Linear/Embedding 权重），no_decay 只有 bias 和 LayerNorm。")


decay 参数数量:    6, 总元素: 193536
no_decay 参数数量: 9, 总元素: 1896

典型分布：decay 占绝大多数（Linear/Embedding 权重），no_decay 只有 bias 和 LayerNorm。


### 7.3 Gradient Clipping

训练初期，模型参数离最优解很远，loss surface 的曲率不可预测。一次 outlier batch（数据特别脏、序列特别长）就可能产生极大梯度，把参数一把推飞。Gradient clipping 的作用是给梯度范数设上限，防止任何单步更新灾难性地大。

LLM 里最常用的是 global norm clip：先计算所有参数梯度的 global L2 norm $\|g\| = \sqrt{\sum_i \|g_i\|^2}$，如果超过阈值 $\tau$，就把所有梯度按 $\tau / \|g\|$ 等比缩小：

$$g \leftarrow g \cdot \min\left(1, \frac{\tau}{\|g\|}\right)$$

这种做法保留了各参数梯度的相对方向，只整体缩放幅度。阈值 $\tau$ 在 GPT-3、LLaMA 等大模型里通常设为 1.0。

Transformer 特别需要 clip：初期 attention 的 softmax 还没收敛，少量异常激活可能让某些 head 的梯度比平均值大几个数量级；加上残差连接会把这种异常逐层放大。没有 clip，一次 bad batch 可能让整层参数飞掉。


In [12]:
import torch


def clip_grad_norm_(params, max_norm, eps=1e-6):
    """手写 global norm clip，等价于 torch.nn.utils.clip_grad_norm_

    参数:
        params: 可迭代的参数（grad 不能为 None）
        max_norm: 梯度 global norm 的上限
        eps: 防 0 的小常数
    """
    grads = [p.grad for p in params if p.grad is not None]
    if len(grads) == 0:
        return torch.tensor(0.0)

    # 计算 global norm：所有梯度 L2 norm 平方之和再开方
    total_norm_sq = sum(g.float().pow(2).sum() for g in grads)
    total_norm = total_norm_sq.sqrt()

    # 如果超过阈值，等比缩小所有梯度
    clip_coef = max_norm / (total_norm + eps)
    if clip_coef < 1:
        for g in grads:
            g.mul_(clip_coef)

    return total_norm


# 验证：构造一组梯度，global norm 远超 1.0
params = [torch.randn(100, requires_grad=True) for _ in range(3)]
for p in params:
    p.grad = torch.randn(100) * 10  # 故意放大梯度

before_norm = torch.sqrt(sum(p.grad.float().pow(2).sum() for p in params)).item()
print(f"clip 前的 global norm: {before_norm:.2f}")

returned = clip_grad_norm_(params, max_norm=1.0)
after_norm = torch.sqrt(sum(p.grad.float().pow(2).sum() for p in params)).item()
print(f"clip 后的 global norm: {after_norm:.4f}")
print()
print("关键观察：clip 后 global norm 被压到 1.0 附近，但梯度方向被保留。")
print("clip 不会放大梯度——原始 norm 小于阈值时，clip_coef >= 1 但不应用。")


clip 前的 global norm: 180.46
clip 后的 global norm: 1.0000

关键观察：clip 后 global norm 被压到 1.0 附近，但梯度方向被保留。
clip 不会放大梯度——原始 norm 小于阈值时，clip_coef >= 1 但不应用。


### 7.4 Muon：矩阵正交化的梯度更新

Muon（Keller Jordan, 2024）的核心观察：Linear 层的权重是一个 2D 矩阵 $W \in \mathbb{R}^{m \times n}$，对应的梯度 $G$ 也是 2D。Adam 把 $G$ 展平成向量，逐元素做 $m/\sqrt{v}$，丢掉了矩阵结构。如果把 $G$ 当成矩阵处理，可以用 Newton-Schulz 迭代把它正交化，得到一个方向更稳、步长更可控的更新。

正交化的直觉：把梯度 $G$ 做 SVD 分解 $G = U \Sigma V^\top$，正交化就是把奇异值全部替换成 1，得到 $U V^\top$。这样更新方向的幅度被归一化，只有方向被保留——类似于 sign-gradient，但保留了矩阵的所有方向信息。

直接做 SVD 太慢。Newton-Schulz 迭代是一个近似正交化的迭代算法：

$$X_{k+1} = \frac{1}{2} X_k (3 I - X_k^\top X_k)$$

从 $X_0 = G / \|G\|_F$ 出发，迭代 7-10 次就能让 $X_k$ 接近 $G$ 的正交化版本 $U V^\top$。每次迭代只需要几次矩阵乘法，比 SVD 快一个数量级。

Muon 只对 2D 矩阵参数用。1D 参数（bias、LayerNorm）没有矩阵结构，正交化无意义，仍然用 Adam。所以实际使用时是「Muon for 2D + Adam for 1D」的混合优化器。


In [13]:
import torch

torch.manual_seed(42)


def newton_schulz(G, steps=7):
    """用 Newton-Schulz 迭代近似把 G 正交化

    参数:
        G: 2D 梯度矩阵
        steps: 迭代次数，7-10 次通常足够
    返回:
        近似正交化后的矩阵，形状与 G 相同
    """
    assert G.dim() == 2, "Newton-Schulz 只适用于 2D 矩阵"
    X = G.bfloat16() if G.is_cuda else G.float()
    # 归一化：让谱半径 < 1.5，保证迭代收敛
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.t()
        B = A @ X
        X = 1.5 * X - 0.5 * B
    return X


# 验证 Newton-Schulz 的正交化效果
G = torch.randn(8, 16)
X = newton_schulz(G, steps=7)

# 正交化的判据：X X^T 接近单位矩阵
gram = X @ X.t()
identity = torch.eye(8)
off_diag = (gram - identity).abs().max().item()
diag_dev = (gram.diag() - 1.0).abs().max().item()

print(f"输入矩阵 G 形状: {tuple(G.shape)}")
print(f"迭代 7 次后 X X^T 的对角线偏离 1: {diag_dev:.2e}")
print(f"迭代 7 次后 X X^T 的非对角线最大值:   {off_diag:.2e}")
print()
print("关键观察：Newton-Schulz 迭代 7 步后，X X^T 已非常接近单位矩阵。")
print("等价于把 G 的 SVD 奇异值全部替换为 1，保留了方向但归一化了幅度。")


输入矩阵 G 形状: (8, 16)
迭代 7 次后 X X^T 的对角线偏离 1: 3.27e-03
迭代 7 次后 X X^T 的非对角线最大值:   3.27e-03

关键观察：Newton-Schulz 迭代 7 步后，X X^T 已非常接近单位矩阵。
等价于把 G 的 SVD 奇异值全部替换为 1，保留了方向但归一化了幅度。


#### 简化版 Muon 优化器

把 Newton-Schulz 嵌进 `step()`，加上 momentum，就是一个最小可用的 Muon。教学版省略了原版的 RMS scaling、参数形状处理等工程细节，但核心思想完整。原版 Muon 仓库在 [KellerJordan/Muon](https://github.com/KellerJordan/Muon)，nanoGPT Speedrun 里用 Muon 把训练时间从 45 分钟降到 22 分钟。

Muon 的学习率通常比 Adam 大一个数量级（$4 \times 10^{-3}$ vs $3 \times 10^{-4}$），因为正交化后的更新方向幅度被严格控制，可以放心走大步。


In [14]:
import torch


class SimpleMuon(torch.optim.Optimizer):
    """简化版 Muon：对 2D 参数用 Newton-Schulz 正交化，对 1D 退化为 momentum SGD

    参数:
        params: 待优化参数
        lr: 学习率（Muon 通常 4e-3，比 Adam 大一个数量级）
        momentum: 一阶矩衰减系数
        ns_steps: Newton-Schulz 迭代次数
    """

    def __init__(self, params, lr=4e-3, momentum=0.95, ns_steps=5):
        defaults = {"lr": lr, "momentum": momentum, "ns_steps": ns_steps}
        super().__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            momentum = group["momentum"]
            ns_steps = group["ns_steps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state["momentum_buffer"] = torch.zeros_like(p)

                buf = state["momentum_buffer"]
                buf.mul_(momentum).add_(grad)

                if p.dim() >= 2:
                    # 2D 参数：用 Newton-Schulz 正交化 momentum
                    update = newton_schulz(buf, steps=ns_steps)
                    # 按 sqrt(max(m, n)) 缩放，匹配原版 Muon 的幅度控制
                    scale = max(1.0, buf.shape[0] / buf.shape[1]) ** 0.5
                    p.data.add_(update, alpha=-lr * scale)
                else:
                    # 1D 参数：退化为普通 momentum SGD
                    p.data.add_(buf, alpha=-lr)


# 冒烟测试：Muon 能跑、参数会变
torch.manual_seed(0)
p = torch.randn(32, 64, requires_grad=True)
p.grad = torch.randn(32, 64) * 0.1
before = p.data.clone()
opt = SimpleMuon([p], lr=4e-3, ns_steps=7)
opt.step()
delta = (p - before).abs().mean().item()
print(f"参数平均更新幅度: {delta:.6f}")
print(f"参数维度: {tuple(p.shape)}")
print("关键观察：Muon 对 2D 参数应用 Newton-Schulz 正交化后的更新方向。")


参数平均更新幅度: 0.000390
参数维度: (32, 64)
关键观察：Muon 对 2D 参数应用 Newton-Schulz 正交化后的更新方向。


### 7.5 Lion / Shampoo / SOAP 谱系

Adam 和 Muon 之外，2023-2025 年还有几条值得知道的优化器路线。它们的共同动机是：Adam 把每个参数位置独立看待，忽略了梯度作为矩阵/张量的结构信息。

**Lion**（Google, 2023）的更新规则极其简单：$\theta \leftarrow \theta - \eta \cdot \text{sign}(m)$，其中 $m$ 是 momentum。和 Adam 的区别是 Lion 不算二阶矩，直接取 sign——这等价于一个极度激进的自适应学习率。Lion 省掉了 $v$ 的存储，每个参数少占一份显存；在 Vision 和 LLM 上都报告了和 AdamW 相当甚至更好的结果。代价是对学习率和 weight decay 的配比很敏感。

**Shampoo**（Gupta et al., 2018）是真正的二阶优化器。它维护两个预处理矩阵 $L = \mathbb{E}[g g^\top]$（左乘）和 $R = \mathbb{E}[g^\top g]$（右乘），用 $L^{-1/4} G R^{-1/4}$ 作为更新方向。这是 Adam 在矩阵情形下的自然推广——Adam 对角化协方差矩阵，Shampoo 做完整矩阵预处理。问题是 $L$、$R$ 的存储和求逆开销巨大，长期只在小模型上使用。

**SOAP**（Shi et al., 2024）是 Shampoo 的工程优化版。它把 $L$、$R$ 的更新放在低频路径上（每 N 步才更新一次），并复用 Adam 的逐元素状态做高频修正。SOAP 在 LLaMA 规模上报告了比 AdamW 快 30%-50% 的收敛速度。

这三个优化器和 Adam/Muon 的核心区别在于对梯度矩阵结构信息的利用程度：

| 优化器 | 矩阵结构利用 | 存储开销 | 收敛速度 | 社区成熟度 |
|:---|:---|:---|:---|:---|
| Adam/AdamW | 无（逐元素） | 2× 参数量 | 基线 | 最成熟 |
| Lion | 无（sign） | 1× 参数量 | 相当或更好 | 中等 |
| Muon | 有（正交化） | 1× 参数量 | 比 Adam 快 | 早期 |
| Shampoo | 有（完整二阶） | 大矩阵 | 理论上更快 | 低 |
| SOAP | 有（低频二阶+逐元素修正） | 中等 | 比 Adam 快 30-50% | 早期 |

一个常见的工程误区是看到新优化器就想换。AdamW 经过 GPT-3、LLaMA、Qwen、DeepSeek 等几十个千亿模型验证，超参空间被充分探索。换成 Muon 或 SOAP 需要重新调学习率、weight decay、warmup，实验成本不低。除非有明确的吞吐量瓶颈，AdamW + 良好的 LR schedule 是最稳妥的选择。


### 7.6 优化器选型小结

不同训练阶段的优化器选择和建议：

| 训练阶段 | 推荐优化器 | 典型学习率 | 备注 |
|:---|:---|:---|:---|
| 预训练（dense） | AdamW | $3 \times 10^{-4}$ | 社区调参最充分，首选 |
| 预训练（MoE） | AdamW | $3 \times 10^{-4}$ | router 单独设更小 lr |
| SFT / 指令微调 | AdamW | $10^{-5}$ 到 $10^{-4}$ | weight decay 通常设为 0 |
| RLHF（PPO） | AdamW | actor 更小 | actor/critic 不同 lr |
| 中等规模实验 | Muon | $4 \times 10^{-3}$ | 2D 用 Muon，1D 用 Adam |

切换优化器必须重新调学习率——Adam 的 lr 不能直接用到 Muon 上，反过来也一样。


## 8. 把迷你 Trainer 映射到 Hugging Face Transformers

如果换成 Transformers，代码大概长这样：

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM，不是 BERT 那种 masked LM
)

args = TrainingArguments(
    output_dir="outputs/qwen-sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=500,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collator,
)

trainer.train()
```

和我们手写 loop 的对应关系是：

| 我们手写的组件 | Transformers 里的组件 |
|:---|:---|
| `ToyTextDataset` | `Dataset` / `datasets.Dataset` |
| `simple_collate` | `DataCollatorForLanguageModeling` 或自定义 collator |
| `TinyCausalLM` | `AutoModelForCausalLM` |
| `optimizer` | Trainer 内部创建的 AdamW 或你自定义的 optimizer |
| `for epoch... for batch...` | `trainer.train()` |
| `outputs = model(**batch)` | Trainer 内部的 `training_step` |
| `loss.backward()` | Trainer / Accelerate 处理 backward |
| `optimizer.step()` | Trainer 内部按梯度累积节奏 step |

注意一个细节：纯文本继续预训练（CPT）和对话 SFT 的 collator 不完全一样。

| 训练类型 | labels 通常怎么算 |
|:---|:---|
| 继续预训练 / Causal LM | 基本所有非 PAD token 都算 loss |
| 指令微调 / SFT | user/system 部分多半设为 `-100`，主要训练 assistant 回复 |


## 9. 把同一条链路映射到 ModelScope ms-swift

你提到的 “ms-xxx” 通常就是 **ModelScope SWIFT / `ms-swift`**。

它的定位可以粗略理解为：面向大模型和多模态模型的训练工具箱。不需要自己编写 `Trainer`、LoRA 注入、模板处理、DeepSpeed 配置和评测脚本，这些组件被自动整合。

常见命令形态类似：

```bash
swift sft \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --dataset your_dataset.jsonl \
  --train_type lora \
  --learning_rate 2e-5 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 8 \
  --num_train_epochs 3 \
  --output_dir output/qwen-lora
```

这行命令背后的概念，仍然是我们刚刚写过的 loop：

| `ms-swift` 参数/功能 | 背后的训练环节 |
|:---|:---|
| `--model` | 加载模型和 tokenizer |
| `--dataset` | 读取原始样本，转成训练 dataset |
| 模型 template | 把 messages 转成模型需要的 Chat Template |
| `--train_type lora` | 只给部分低秩参数打开梯度 |
| `--per_device_train_batch_size` | 每张卡一次喂多少条样本 |
| `--gradient_accumulation_steps` | 几个小 batch 合成一次大更新 |
| `--learning_rate` | optimizer 的步长 |
| `--deepspeed` / 分布式参数 | 多卡切分模型状态、梯度、优化器状态 |
| `--output_dir` | 保存 checkpoint 和日志 |

在中国区使用较广泛，因为它与 ModelScope 模型、数据集、Qwen 系列、国内网络环境和常见训练配置结合更紧密。

但要记住：`ms-swift` 不是另一个“训练原理”。它是工程封装。真正发生的仍然是：

```text
batch -> model forward -> loss -> backward -> optimizer step
```


## 10. 工业训练 loop 的完整流程图

把本章内容压缩成一张图：

```text
1. 原始数据
   纯文本："我爱机器学习"
   对话：[{role: user, content: ...}, {role: assistant, content: ...}]

2. 模板化
   apply_chat_template(messages)
   -> "<user>...<assistant>..."

3. Tokenize
   tokenizer(text)
   -> input_ids

4. 构造 labels
   继续预训练：labels ≈ input_ids 的下一个 token
   SFT：user/system 位置设为 -100，assistant 位置保留答案

5. Data Collator
   多条样本 pad 到同一长度
   -> input_ids / attention_mask / labels

6. Forward
   outputs = model(**batch)
   -> logits: [batch, seq_len, vocab_size]
   -> loss: 有效 token 的平均 cross-entropy

7. Backward
   loss.backward()
   -> 每个可训练参数得到梯度

8. Optimizer Step
   AdamW / 8-bit Adam / 其他 optimizer
   -> 更新参数

9. Scheduler / Logging / Eval / Checkpoint
   学习率调整、记录 loss、跑验证集、保存权重
```

如果用了 LoRA，再加一句：

```text
原模型大部分参数 frozen，只训练 LoRA 的 A/B 小矩阵。
```

如果用了 DeepSpeed ZeRO / FSDP，再加一句：

```text
参数、梯度、优化器状态被切到多张 GPU 上，但数学上的 loss/backward/step 不变。
```


## 11. Loss 的信息论视角：perplexity、entropy 和 KL

前面手算 CE 时，我们只关心「正确 token 的概率有多低」。但论文和训练日志里还会出现另外几个词：perplexity（PPL）、entropy、KL divergence。它们和 CE 是同一族概念，工程上经常互换使用。

这一节把这些概念串起来：先看 perplexity 怎么从 CE 换算，再看 CE / entropy / KL 之间的等式，最后看这些等式在蒸馏、RLHF、DPO、MoE load balancing 里被复用到哪些地方。


### 11.1 Perplexity：把 loss 翻译成「模型在几个 token 之间犹豫」

Cross-Entropy loss 是一个对数刻度的数字，单看数值不容易建立直觉。Perplexity（PPL）是 CE 的指数化版本：

$$ \text{PPL} = \exp(\text{CE}) $$

PPL 的物理含义可以这样理解：模型在每个位置上，等价于在多少个 token 之间均匀地犹豫。

几个具体数字建立直觉：

| CE loss | PPL | 含义 |
|:---|:---|:---|
| 0 | 1.0 | 模型完全确定正确 token，没有犹豫 |
| 2.0 | 7.39 | 模型在每个位置平均在 7-8 个 token 之间犹豫 |
| 4.6 | 100 | 犹豫范围扩大到约 100 个 token |
| $\log V$ | $V$ | 模型对词表里每个 token 给出均匀分布，等价于瞎猜 |

其中 $V$ 是词表大小。PPL = 词表大小意味着模型预测能力等于随机猜，这是 PPL 的上界。

一个容易踩的坑：PPL 跨数据集通常不可比。两份语料的词表、token 边界、领域难度不同，CE 的绝对值就不一样，直接对比 PPL 数值会得出错误结论。论文里报 PPL 时，通常只和自己用同一份数据集的其他模型比。


In [15]:
# 用具体数字看 CE 和 PPL 之间的换算
import math

ce_values = [0.0, 1.0, 2.0, 3.0, 4.6, math.log(50000)]

print(f"{'CE':>6} | {'PPL':>12} | 直觉")
print("-" * 50)
for ce in ce_values:
    ppl = math.exp(ce)
    if ce == 0.0:
        note = "完全确定，没有犹豫"
    elif ce == math.log(50000):
        note = f"等于在 50000 词表上瞎猜"
    else:
        note = f"平均在约 {ppl:.1f} 个 token 之间犹豫"
    print(f"{ce:>6.2f} | {ppl:>12.2f} | {note}")

print()
print("关键观察：PPL 把对数尺度的 loss 翻译成线性尺度，更接近直觉。")
print("但 PPL 跨数据集通常不可比，必须在同一份语料和同一个 tokenizer 下比较。")


    CE |          PPL | 直觉
--------------------------------------------------
  0.00 |         1.00 | 完全确定，没有犹豫
  1.00 |         2.72 | 平均在约 2.7 个 token 之间犹豫
  2.00 |         7.39 | 平均在约 7.4 个 token 之间犹豫
  3.00 |        20.09 | 平均在约 20.1 个 token 之间犹豫
  4.60 |        99.48 | 平均在约 99.5 个 token 之间犹豫
 10.82 |     50000.00 | 等于在 50000 词表上瞎猜

关键观察：PPL 把对数尺度的 loss 翻译成线性尺度，更接近直觉。
但 PPL 跨数据集通常不可比，必须在同一份语料和同一个 tokenizer 下比较。


### 11.2 CE、KL 和 entropy 之间的等式

Cross-Entropy、KL divergence 和 entropy 三者共享一个恒等式：

$$ H(p, q) = H(p) + D_{KL}(p \| q) $$

其中 $p$ 是真实分布，$q$ 是模型预测分布。$H(p, q)$ 是 cross-entropy，$H(p)$ 是真实分布的 entropy，$D_{KL}(p \| q)$ 是 KL 散度，衡量两个分布的差异。

在 teacher forcing 的语言模型训练里，「真实分布」是 one-hot：正确 token 概率为 1，其他 token 概率为 0。one-hot 分布的 entropy 为 0：

$$ H(p) = -\sum_i p_i \log p_i = -1 \cdot \log 1 = 0 $$

所以 teacher forcing 下有：

$$ \text{CE}(p, q) = D_{KL}(p \| q) $$

这就是为什么训练语言模型时，CE loss 和 KL divergence 的数值完全一致。这并不是巧合，而是 one-hot 监督把 entropy 项归零后的结果。



In [16]:
# 用数值验证：teacher forcing 下 CE == KL
import torch
import torch.nn.functional as F

# 词表大小为 5，正确 token 是 id=2
p = torch.tensor([0.0, 0.0, 1.0, 0.0, 0.0])  # one-hot 真实分布
logits = torch.tensor([1.2, 0.5, 2.8, -0.3, 0.1])
q = F.softmax(logits, dim=-1)  # 模型预测分布

# Cross-Entropy: -sum p_i * log q_i
log_q = torch.log(q)
ce = -(p * log_q).sum().item()

# entropy of p: -sum p_i * log p_i
# p 里只有 id=2 是 1，log(1)=0，所以 H(p) = 0
entropy_p = -(p * torch.log(p.clamp_min(1e-12))).sum().item()

# KL(p || q) = -sum p_i * log(q_i / p_i)
kl = (p * (torch.log(p.clamp_min(1e-12)) - log_q)).sum().item()

print(f"H(p)      = {entropy_p:.4f}")
print(f"CE(p, q)  = {ce:.4f}")
print(f"KL(p||q)  = {kl:.4f}")
print()
print(f"验证 CE == H(p) + KL: {entropy_p:.4f} + {kl:.4f} = {entropy_p + kl:.4f}")
print("关键观察：teacher forcing 下真实分布是 one-hot，H(p)=0，所以 CE 和 KL 数值一致。")


H(p)      = -0.0000
CE(p, q)  = 0.3467
KL(p||q)  = 0.3467

验证 CE == H(p) + KL: -0.0000 + 0.3467 = 0.3467
关键观察：teacher forcing 下真实分布是 one-hot，H(p)=0，所以 CE 和 KL 数值一致。


### 11.3 这些等式在哪里被复用

CE、KL、entropy 不只是预训练 loss 的工具，它们在不同训练范式里反复出现，只是名字和用法换了。

| 训练范式 | 用的形式 | 为什么用这个 |
|:---|:---|:---|
| 预训练 / SFT | CE loss | 监督信号是 one-hot，CE = KL，简单直接 |
| 知识蒸馏 | KL divergence | teacher 是软标签分布，不再是 one-hot，要衡量两个完整分布的差异 |
| RLHF (PPO) | KL penalty | 在 reward 之外加一项 KL(policy \| reference)，防止策略跑偏 |
| DPO | log-ratio（隐含 KL） | 通过 log σ(β log(π/π_ref)) 的形式，等价于一种隐式 KL 约束 |
| MoE load balancing | entropy / KL 形式 | 鼓励 router 把 token 均匀分到各 expert，避免少数 expert 被打爆 |

MoE 的 load balancing loss 是一个典型例子：它把每个 token 对所有 expert 的路由概率看成分布 $r$，希望 $r$ 在整个 batch 上接近均匀。具体实现有的用 entropy 项（让 router 分布的 entropy 尽量大），有的写成 KL($r$ \| uniform)，本质都是同一族操作。

记住这条线索：只要训练目标涉及「让一个分布靠近另一个分布」，CE、KL、entropy 这三件套就会出现。teacher forcing 是最特殊的一例——one-hot 监督让三者数值相等，所以预训练 loss 才能直接写成 CE 的简洁形式。


## 12. 一个容易混淆的问题：模型内部 shift 还是数据里 shift？

有些教程会写：

```python
input_ids = tokens[:-1]
labels = tokens[1:]
```

但很多 Transformers 模型训练时又会写：

```python
input_ids = tokens
labels = tokens
```

这不是矛盾。区别在于 **shift 发生在哪里**。

| 写法 | shift 发生位置 | 适合讲解什么 |
|:---|:---|:---|
| `input=tokens[:-1]`, `labels=tokens[1:]` | 数据处理阶段 | 最适合教学，关系最清楚 |
| `input=tokens`, `labels=tokens` | 模型内部算 loss 时 shift | 工业库常见，省掉手动切片 |

很多 `AutoModelForCausalLM` 在内部算 loss 时，会把 logits 和 labels 对齐成：

```text
shift_logits = logits[..., :-1, :]
shift_labels = labels[..., 1:]
```

所以你看到 `labels=input_ids` 时，不要误会成“模型预测自己”。真正比较的仍然是：当前位置预测下一个 token。


In [17]:
# 对比：数据外部 shift 和模型内部 shift 得到同一种监督关系
full_tokens = torch.tensor([[1, 3, 4, 5, 6, 2]])

external_input = full_tokens[:, :-1]
external_labels = full_tokens[:, 1:]

internal_input = full_tokens
internal_labels = full_tokens
shifted_labels_inside_model = internal_labels[:, 1:]

print("外部 shift:")
print("input: ", external_input.tolist())
print("labels:", external_labels.tolist())
print()

print("内部 shift 的等价结果:")
print("input 传完整 tokens:       ", internal_input.tolist())
print("模型内部拿 labels[:, 1:]:", shifted_labels_inside_model.tolist())
print()

same = torch.equal(external_labels, shifted_labels_inside_model)
print("两种写法的预测目标一致吗？", same)
print("关键观察：工业库常把 shift 藏在模型的 loss 计算里。")


外部 shift:
input:  [[1, 3, 4, 5, 6]]
labels: [[3, 4, 5, 6, 2]]

内部 shift 的等价结果:
input 传完整 tokens:        [[1, 3, 4, 5, 6, 2]]
模型内部拿 labels[:, 1:]: [[3, 4, 5, 6, 2]]

两种写法的预测目标一致吗？ True
关键观察：工业库常把 shift 藏在模型的 loss 计算里。


## 作业

> 可以让 AI 帮忙解释思路、拆步骤、检查方向，但不建议直接让 AI “做完这道题”。


**作业 1：构造 SFT labels**

给定一条对话已经被模板化成 token：

```text
tokens = ["<user>", "你好", "<assistant>", "你好", "呀", "<eos>"]
ids    = [1,       10,   2,            10,   11,   3]
```

请构造 `labels`：只让 assistant 回复 `["你好", "呀", "<eos>"]` 参与 loss，其余位置设为 `-100`。

小提示：assistant 回复从 id 10 开始，到 EOS id 3 结束。


In [18]:
# 作业 1：构造 SFT labels
ids = [1, 10, 2, 10, 11, 3]

# TODO: 只保留 assistant 回复部分的 label，其余位置为 -100
labels = [-100, -100, -100, 10, 11, 3]

assert labels is not None, "请先构造 labels"
assert labels == [-100, -100, -100, 10, 11, 3], f"labels 不对，你得到 {labels}"

print("✅ 作业 1 通过：")
print("   你已经会把 user/template 部分 mask 掉，只训练 assistant 回复。")


✅ 作业 1 通过：
   你已经会把 user/template 部分 mask 掉，只训练 assistant 回复。


**作业 2：计算梯度累积后的等效 batch size**

训练参数如下：

```text
per_device_train_batch_size = 2
num_gpus = 4
gradient_accumulation_steps = 8
```

请计算一次 optimizer step 等效看到了多少条样本。

小提示：等效 batch size = 每卡 batch × GPU 数 × 梯度累积步数。


In [19]:
# 作业 2：计算等效 batch size
per_device_train_batch_size = 2
num_gpus = 4
gradient_accumulation_steps = 8

# TODO: 计算一次 optimizer.step() 前累计了多少条样本
effective_batch_size = per_device_train_batch_size * num_gpus * gradient_accumulation_steps

assert effective_batch_size is not None, "请先计算 effective_batch_size"
assert effective_batch_size == 64, f"答案应为 64，你得到 {effective_batch_size}"

print("✅ 作业 2 通过：")
print(f"   等效 batch size = {effective_batch_size}")
print("   这就是显存不够时常用 gradient accumulation 的原因。")


✅ 作业 2 通过：
   等效 batch size = 64
   这就是显存不够时常用 gradient accumulation 的原因。


**作业 3：只平均有效 token 的 loss**

某个 batch 摊平后有 6 个位置，每个位置的 loss 是：

```text
losses = [0.2, 0.4, 1.0, 0.3, 0.8, 0.5]
labels = [10, 11, -100, 3, -100, 12]
```

请计算有效位置的平均 loss。`labels == -100` 的位置要忽略。

小提示：有效位置是 0、1、3、5。


In [20]:
# 作业 3：只平均有效 token 的 loss
losses = [0.2, 0.4, 1.0, 0.3, 0.8, 0.5]
labels = [10, 11, -100, 3, -100, 12]

# TODO: 过滤掉 labels == -100 的位置，再求平均
valid_average_loss = sum(loss for loss, label in zip(losses, labels) if label != -100) / sum(label != -100 for label in labels)

assert valid_average_loss is not None, "请先计算 valid_average_loss"
expected = (0.2 + 0.4 + 0.3 + 0.5) / 4
assert abs(valid_average_loss - expected) < 1e-6, f"答案应为 {expected:.4f}"

print("✅ 作业 3 通过：")
print(f"   有效平均 loss = {valid_average_loss:.4f}")
print("   你已经理解了 ignore_index=-100 的核心作用。")


✅ 作业 3 通过：
   有效平均 loss = 0.3500
   你已经理解了 ignore_index=-100 的核心作用。


## 小结（checklist）

确认你已经理解下面这些点：

1. ✅ `Trainer` / `ms-swift` 不是魔法，本质是管理训练 loop 的工程封装
2. ✅ 一条样本会经历：模板化 → tokenize → 构造 labels → collator 拼 batch → model forward → loss
3. ✅ Causal LM 的核心目标仍然是 next-token prediction
4. ✅ `logits` 是每个位置对整个词表的打分，Cross-Entropy 只惩罚正确 token 概率不够高
5. ✅ `-100` 表示这个位置不参与 loss，常用于 PAD 和 SFT 中的 user/system 部分
6. ✅ `attention_mask` 管模型"看不看 pad"，`labels=-100` 管 loss"算不算 pad"
7. ✅ Hugging Face Transformers 的 `Trainer` 和 ModelScope `ms-swift` 都是在这条主线上做工程增强
8. ✅ `gradient_accumulation_steps` 改变的是多久更新一次参数，不改变单次 forward 的 batch 形状
9. ✅ LoRA 只改变"哪些参数可训练"，不改变 loss/backward/optimizer step 的基本逻辑
10. ✅ DeepSpeed/FSDP 改变参数和优化器状态如何分布在多卡上，不改变训练的数学目标
11. ✅ Adam 维护一阶矩 $m$（momentum）和二阶矩 $v$（variance），bias correction 修正初期估计偏差
12. ✅ AdamW 把 weight decay 从梯度链路解耦；bias 和 LayerNorm 不加 weight decay，用 parameter group 区分
13. ✅ Gradient clipping（global norm clip，$\\tau = 1.0$）防止单步更新过大，保留方向只缩放幅度
14. ✅ Muon 用 Newton-Schulz 迭代把梯度矩阵正交化，保留方向、归一化幅度，只对 2D 参数使用
15. ✅ AdamW 仍然是大模型预训练的事实标准，切换优化器必须重新调学习率

一句话总结：工业训练库把大量工程细节包起来，但你只要抓住 `batch -> model -> loss -> backward -> step` 这条主线，就不会被 `trainer.train()` 吓住。

下一节会进入 Scaling Laws：当你知道怎么训练以后，下一个问题就是"模型多大、数据多少、算力多少才划算"。